# World Cup xG Lab: Shot Data Exploration

This notebook explores the processed StatsBomb shot dataset before modeling. The goal is to understand the size of the data, missing values, scoring rate, team/player coverage, shot outcomes, and basic shot locations.

## 1. Load the Dataset

We start by loading `data/processed/shots.csv`, which should be created by the StatsBomb ingestion script.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SHOTS_FILE = PROJECT_ROOT / "data" / "processed" / "shots.csv"

shots = pd.read_csv(SHOTS_FILE)
shots.head()

## 2. Dataset Shape and Columns

The shape tells us how many shot rows and columns are available. The column list shows which fields we can use for EDA and later feature engineering.

In [ ]:
print(f"Rows: {shots.shape[0]:,}")
print(f"Columns: {shots.shape[1]:,}")

shots.columns.tolist()

## 3. Missing Values

Missing values help us identify fields that need cleaning, filtering, or careful handling before modeling.

In [ ]:
missing_values = shots.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

## 4. Goal Rate

The goal rate is the share of shots that became goals. This is the baseline rate for the target variable in an expected goals model.

In [ ]:
goal_rate = shots["is_goal"].mean()
print(f"Goal rate: {goal_rate:.2%}")

## 5. Top 10 Teams by Shot Count

This shows which teams appear most often in the shot dataset. Large differences can reveal competition or team coverage imbalance.

In [ ]:
top_teams = shots["team"].value_counts().head(10)
top_teams

## 6. Top 10 Players by Shot Count

This shows the players with the most shots. It can help identify whether a few high-volume players dominate the dataset.

In [ ]:
top_players = shots["player"].value_counts().head(10)
top_players

## 7. Shot Outcome Distribution

Shot outcomes show how often shots are goals, saved, blocked, off target, or have other results.

In [ ]:
shot_outcomes = shots["shot_outcome"].value_counts()
shot_outcomes

## 8. Body Part Distribution

If `body_part` is available, this shows whether shots are mostly taken with feet, headers, or other body parts.

In [ ]:
if "body_part" in shots.columns:
    display(shots["body_part"].value_counts())
else:
    print("body_part column is not available.")

## 9. Basic Shot Location Plot

This scatter plot shows where shots were taken from. StatsBomb coordinates use a 120 by 80 pitch, with shots usually moving toward the right-side goal.

In [ ]:
plot_data = shots.dropna(subset=["shot_x", "shot_y"])

plt.figure(figsize=(10, 6))
plt.scatter(
    plot_data["shot_x"],
    plot_data["shot_y"],
    c=plot_data["is_goal"].map({True: "tab:green", False: "tab:blue"}),
    alpha=0.25,
    s=12,
)
plt.xlim(0, 120)
plt.ylim(0, 80)
plt.xlabel("Shot X")
plt.ylabel("Shot Y")
plt.title("Shot Locations: Goals in Green, Other Shots in Blue")
plt.grid(alpha=0.2)
plt.show()

## 10. Quick Takeaways

Before modeling, look for missing important fields, unusual class balance, heavy team/player imbalance, and whether shot locations match football intuition. Good early xG features often come from shot location, angle, body part, play pattern, pressure, and shot type.